# Analyze eBPF Programs

This notebook compiles, analyzes, and generates the report for eBPF programs. 

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from depsurf import system
from utils import SOFTWARE_PATH

BCC_PATH = SOFTWARE_PATH / "bcc"
BCC_TOOLS_PATH = BCC_PATH / "libbpf-tools"
BCC_OBJ_PATH = BCC_TOOLS_PATH / ".output"

TRACEE_PATH = SOFTWARE_PATH / "tracee"
TRACEE_OBJ_FILE = TRACEE_PATH / "dist" / "tracee.bpf.o"

The following cell compiles the BCC tools. 

In [ ]:
system(f"make -C {BCC_TOOLS_PATH} -j $(nproc)", linux=True)

The following cell compiles the eBPF program for Tracee.

In [ ]:
system(f"make -C {TRACEE_PATH} bpf -j $(nproc)", linux=True)

The following cell analyzes and generates the report for eBPF programs.

In [ ]:
from depsurf import BPFProgram, VersionGroup, DepReport, DepKind
from utils import OUTPUT_PATH, save_pkl

GROUPS = [VersionGroup.REGULAR, VersionGroup.ARCH]
PROG_PATHS = sorted(BCC_OBJ_PATH.glob("*.bpf.o")) + [TRACEE_OBJ_FILE]


REPORT_KINDS = [
    DepKind.FUNC,
    DepKind.TRACEPOINT,
    DepKind.LSM,
    DepKind.FIELD,
    DepKind.STRUCT,
    DepKind.SYSCALL,
]

results = {}
for path in PROG_PATHS:
    bpf = BPFProgram(path)
    result_path = OUTPUT_PATH / "programs" / bpf.name
    result_path.mkdir(parents=True, exist_ok=True)
    prog_report = {}
    for dep in bpf.deps:
        if dep.kind not in REPORT_KINDS:
            continue
        with open(result_path / f"{dep.kind} {dep.name}.md", "w") as f:
            report = DepReport.from_groups(dep, GROUPS)
            report.print(file=f)
            prog_report[dep] = report
    print(f"Report saved to {result_path}", flush=True)
    results[bpf.name] = prog_report


save_pkl(results, "bcc")


In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter

results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

rows = list(results.keys())
kinds = list(REPORT_KINDS)
cols = [k.name for k in kinds] + ["TOTAL"]

matrix = []
for prog in rows:
    prog_report = results.get(prog, {})
    counts = Counter(dep.kind for dep in prog_report.keys() if dep.kind in kinds)
    row = [int(counts.get(k, 0)) for k in kinds]
    matrix.append([*row, int(sum(row))])

csv_path = results_dir / "50_programs.csv"
with csv_path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["PROGRAM", *cols])
    for prog, row in zip(rows, matrix):
        w.writerow([prog, *row])

print(f"Saved CSV to: {csv_path}")
